# Practice Exercise: `*args` and `**kwargs` in LLM Function Definitions

## Scenario
You are building a lightweight helper around an LLM client where:
1. You pass an arbitrary number of **system or context instruction blocks** (`*system_notes`) that get joined into a single system prompt.
2. You require a `user_prompt` string.
3. You accept **any extra model or request parameters** (`**model_kwargs`), such as `temperature=0.2`, `max_tokens=500`, `top_p=0.9`, or `response_format={"type": "json_object"}`, without hardcoding every parameter in the signature.

---

## Task 1: `build_llm_payload`
Write a function:
```python
build_llm_payload(user_prompt, *system_notes, default_temperature=0.7, **model_kwargs)

that returns a dictionary: {
    "system": ...,       # Joined string of system_notes, or None if none were provided
    "user": ...,         # The user_prompt
    "parameters": ...    # Dict of model settings (including temperature)
}

In [30]:
def build_llm_payload(user_prompt, *system_notes, default_temperature=0.7, **model_kwargs):
    nonempty_notes = [note.strip() for note in system_notes if note.strip()]
    system = "\n\n".join(nonempty_notes) if nonempty_notes else None

    parameters = dict(model_kwargs)
    parameters.setdefault("temperature", default_temperature)

    return {
        "system": system,
        "user": user_prompt,
        "parameters": parameters,
    }

In [31]:
print(build_llm_payload("Hey LLM, help me with this", *("make it factual", "make it scientific")))

{'system': 'make it factual /n/n make it scientific /n/n ', 'user': 'Hey LLM, help me with this', 'parameters': {'temperature': 0.7}}


In [32]:
payload_1 = build_llm_payload("Summarize this transcript.")
print("Test 1:", payload_1)

Test 1: {'system': '', 'user': 'Summarize this transcript.', 'parameters': {'temperature': 0.7}}


In [34]:
payload_1 = build_llm_payload("Summarize this transcript.")
print("Test 1:", payload_1)

payload_2 = build_llm_payload(
    "Extract action items as JSON.",
    "You are a meeting assistant.",
    "   ",
    "Output strictly JSON.",
    default_temperature=0.3,
    temperature=0.0,
    max_tokens=800,
    model="gpt-4o-mini",
)
print("Test 2:", payload_2)

Test 1: {'system': '', 'user': 'Summarize this transcript.', 'parameters': {'temperature': 0.7}}
Test 2: {'system': 'You are a meeting assistant. /n/n     /n/n Output strictly JSON. /n/n ', 'user': 'Extract action items as JSON.', 'parameters': {'temperature': 0.0, 'max_tokens': 800, 'model': 'gpt-4o-mini'}}
